In [1]:
# 1. Installations and Dependencies

# %pip install -qU google-generativeai
# %pip install -q google-ai-generativelanguage==0.6.15
# %pip install -qU langchain-google-genai
# %pip install -qU langchain-community
# %pip install -qU langgraph
# %pip install -qU python-dotenv


In [1]:
# 2. Environment setup and LLM initialization via LangChain
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env file.")

# Initialize the LLM using LangChain (ready for use with LangGraph + ReAct)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0
)

# Optional test (comment out if not needed)
print("API key loaded:", api_key is not None)
print("Testing LLM:")
print(llm.invoke("Hello, what is your name?").content)

API key loaded: True
Testing LLM:
I do not have a name. I am a large language model, trained by Google.


In [2]:
# 3. Robust LLM wrapper with quota-aware retry logic (LangChain-compatible)
import time
import re
from functools import wraps
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from typing import List, Union, Any, Callable, Optional, Dict

class QuotaAwareLLM:
    """
    Wrapper for LangChain LLMs that adds quota-aware retries.
    Retries on quota/rate-limit errors with fixed delay (no exponential backoff).
    Compatible with LangGraph agent loops.
    """
    
    def __init__(
        self,
        llm: BaseChatModel,
        max_retries: int = 3,
        sleep_on_quota_sec: int = 35
    ):
        self.llm = llm
        self.max_retries = max_retries
        self.sleep_on_quota_sec = sleep_on_quota_sec

    def _is_quota_error(self, exc: Exception) -> bool:
        """Detect quota/rate-limit errors from LangChain-wrapped Google API."""
        error_str = str(exc).lower()
        quota_keywords = ["quota", "rate limit", "429", "resource exhausted", "retry after"]
        return any(kw in error_str for kw in quota_keywords)

    def invoke_with_retry(
        self,
        messages: List[BaseMessage],
        **kwargs
    ) -> BaseMessage:
        """Invoke LLM with retry logic for quota errors."""
        attempt = 0
        while True:
            try:
                return self.llm.invoke(messages, **kwargs)
            except Exception as e:
                attempt += 1
                if attempt > self.max_retries or not self._is_quota_error(e):
                    raise  # Re-raise non-quota errors or exhausted retries
                
                print(f"[QuotaAwareLLM] Quota error detected. "
                      f"Sleeping {self.sleep_on_quota_sec}s (attempt {attempt}/{self.max_retries})...")
                time.sleep(self.sleep_on_quota_sec)
                continue

    def __call__(
        self,
        messages: Union[str, List[BaseMessage]],
        **kwargs
    ) -> str:
        """Simplified interface for agent nodes."""
        if isinstance(messages, str):
            messages = [HumanMessage(content=messages)]
        
        response = self.invoke_with_retry(messages, **kwargs)
        return response.content if hasattr(response, "content") else str(response)

In [3]:
# 4. ReAct system prompt definition
REACT_SYSTEM_PROMPT = """
You are a reasoning agent following the ReAct (Reasoning + Acting) framework.

Workflow:
1. THINK: Analyze the question and plan your approach
2. ACT: Use available tools to gather information
3. OBSERVE: Process tool responses
4. Repeat until ready for final ANSWER

Available tools:
- query_stock_level(item): Check inventory stock for a product
- get_product_price(product): Get current price of a product

Response format:
Thought: [Your reasoning]
Action: tool_name[argument]
PAUSE

Observation: [Tool result will appear here]
Answer: [Final comprehensive answer]

Example:
User: What's the price of a gaming mouse and how many are in stock?
Thought: I need to get the price first, then check stock levels.
Action: get_product_price[gaming mouse]
PAUSE

Observation: PRICE: Gaming Mouse - $99.50 USD per unit
Thought: Now I need to check the inventory.
Action: query_stock_level[gaming mouse]
PAUSE

Observation: Stock available: 80 units of Gaming Mouse
Answer: The gaming mouse costs $99.50 each and we have 80 units in stock.
""".strip()

# Validate prompt integrity (optional but useful for notebooks)
assert "ReAct" in REACT_SYSTEM_PROMPT, "System prompt must contain ReAct framework description"
print(f"[INFO] ReAct system prompt loaded ({len(REACT_SYSTEM_PROMPT)} characters)")

[INFO] ReAct system prompt loaded (1033 characters)


In [4]:
# 5. Agent state definition for ReAct workflow
from typing import TypedDict, List, Optional, Dict, Any

class AgentState(TypedDict):
    """
    State structure for ReAct agent in LangGraph.
    Tracks the full reasoning cycle while remaining serializable.
    """
    input: str                     # Original user query
    messages: List[Dict[str, str]] # Conversation history [{"role": "...", "content": "..."}, ...]
    next_action: Optional[Dict[str, Any]]  # Parsed tool call: {"tool": "name", "args": {"arg": "value"}}
    observation: Optional[str]     # Tool response to be processed
    answer: Optional[str]          # Final answer when workflow completes
    attempts: int                  # Retry counter for quota errors (max 3 attempts)

In [5]:
# 6. Tool implementations with error handling and consistent formatting
import time
from typing import Dict

def query_stock_level(item_name: str) -> str:
    """
    Simulates inventory lookup with consistent ReAct-compatible formatting.
    
    Args:
        item_name: Product name to check stock for (case-insensitive)
    
    Returns:
        Formatted string matching ReAct observation format
    """
    time.sleep(0.1)  # Simulate real API latency
    normalized = item_name.lower().strip().replace("-", " ").replace("_", " ")
    
    # Inventory database with display names
    stock_db: Dict[str, Dict[str, Any]] = {
        "monitor": {"display": "Monitor", "quantity": 75},
        "keyboard": {"display": "Keyboard", "quantity": 120},
        "mouse": {"display": "Mouse", "quantity": 80},
        "gaming mouse": {"display": "Gaming Mouse", "quantity": 80},
        "webcam": {"display": "Webcam", "quantity": 40},
        "headset": {"display": "Headset", "quantity": 60},
        "printer": {"display": "Printer", "quantity": 15},
        "laptop": {"display": "Laptop", "quantity": 25}
    }
    
    item = stock_db.get(normalized)
    if not item:
        return f"Observation: Item '{item_name}' not found in inventory system"
    
    return f"Stock available: {item['quantity']} units of {item['display']}"

def get_product_price(product: str) -> str:
    """
    Simulates price lookup with ReAct-compatible response formatting.
    
    Args:
        product: Product name for price lookup (case-insensitive)
    
    Returns:
        Formatted price string matching system prompt examples
    """
    time.sleep(0.1)  # Simulate real API latency
    normalized = product.lower().strip().replace("-", " ").replace("_", " ")
    
    # Price catalog with display names
    price_db: Dict[str, Dict[str, Any]] = {
        "monitor": {"display": "Monitor", "price": 999.90},
        "keyboard": {"display": "Keyboard", "price": 150.00},
        "mouse": {"display": "Mouse", "price": 45.00},
        "gaming mouse": {"display": "Gaming Mouse", "price": 99.50},
        "webcam": {"display": "Webcam", "price": 120.00},
        "headset": {"display": "Headset", "price": 180.00},
        "printer": {"display": "Printer", "price": 750.00}
    }
    
    item = price_db.get(normalized)
    if not item:
        return f"Observation: Product '{product}' not found in pricing catalog"
    
    return f"PRICE: {item['display']} - ${item['price']:.2f} USD per unit"

In [7]:
# 7. Tool validation tests with expected outputs
print("=== STOCK CHECK TESTS ===")
print(f"Keyboard stock: {query_stock_level('keyboard')}")
print(f"Monitor stock: {query_stock_level('monitor')}")
print(f"Invalid item test: {query_stock_level('smartphone')}\n")

print("=== PRICE CHECK TESTS ===")
print(f"Printer price: {get_product_price('printer')}")
print(f"Gaming mouse price: {get_product_price('gaming mouse')}")
print(f"Invalid product test: {get_product_price('tablet')}\n")

# Automated validation checks
assert "120 units of Keyboard" in query_stock_level("keyboard"), "Keyboard stock test failed"
assert "75 units of Monitor" in query_stock_level("monitor"), "Monitor stock test failed"
assert "not found" in query_stock_level("smartphone"), "Invalid item handling failed"

assert "$750.00 USD per unit" in get_product_price("printer"), "Printer price test failed"
assert "$99.50 USD per unit" in get_product_price("gaming mouse"), "Gaming mouse price test failed"
assert "not found" in get_product_price("tablet"), "Invalid product handling failed"

print("[SUCCESS] All tool validation tests passed!")

=== STOCK CHECK TESTS ===
Keyboard stock: Stock available: 120 units of Keyboard
Monitor stock: Stock available: 75 units of Monitor
Invalid item test: Observation: Item 'smartphone' not found in inventory system

=== PRICE CHECK TESTS ===
Printer price: PRICE: Printer - $750.00 USD per unit
Gaming mouse price: PRICE: Gaming Mouse - $99.50 USD per unit
Invalid product test: Observation: Product 'tablet' not found in pricing catalog

[SUCCESS] All tool validation tests passed!


In [6]:
# 8. Optimized ReAct agent for notebook testing
import re
from typing import Dict, Callable, List

class ReActAgent:
    """
    Notebook-optimized ReAct agent with:
    - Product name normalization
    - Reduced quota sleep times
    - Clearer error handling
    """
    
    def __init__(
        self,
        robust_llm: Callable,
        system_prompt: str,
        tools: Dict[str, Callable]
    ):
        self.robust_llm = robust_llm
        self.system_prompt = system_prompt
        self.tools = tools
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]
        # Product normalization rules (singular/plural mapping)
        self.product_aliases = {
            "keyboards": "keyboard",
            "mice": "mouse",
            "monitors": "monitor",
            "headsets": "headset",
            "printers": "printer",
            "laptops": "laptop",
            "webcams": "webcam"
        }
    
    def normalize_product_name(self, name: str) -> str:
        """Convert plural to singular and standardize names"""
        cleaned = name.lower().strip().replace("-", " ").replace("_", " ")
        return self.product_aliases.get(cleaned, cleaned)
    
    def extract_action(self, response: str) -> str:
        """Strict pattern matching as defined in system prompt"""
        pattern = r"Action:\s*(\w+)\s*\[\s*(.+?)\s*\]"
        if match := re.search(pattern, response):
            return f"{match.group(1).strip()}[{match.group(2).strip()}]"
        return ""
    
    def run(self, query: str, max_iterations: int = 3) -> str:  # Reduced from 5
        """Optimized for notebook execution"""
        print(f"\n🔍 Query: {query}")
        
        self.messages.append({"role": "user", "content": query})
        
        for iteration in range(max_iterations):
            prompt = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in self.messages)
            
            print(f"\n🔄 Iteration {iteration + 1}/{max_iterations}")
            try:
                response = self.robust_llm(prompt)
            except Exception as e:
                print(f"[!] LLM error: {str(e)}")
                return f"AGENT ERROR: {str(e)}"
            
            self.messages.append({"role": "assistant", "content": response})
            print(f"🤖 Response: {response[:200]}...")
            
            # Check for final answer
            if "Answer:" in response:
                return response.split("Answer:", 1)[1].strip()
            
            # Execute action
            if action_text := self.extract_action(response):
                print(f"⚡ Action: {action_text}")
                
                if match := re.match(r"(\w+)\s*\[\s*(.+?)\s*\]", action_text):
                    tool_name = match.group(1)
                    raw_arg = match.group(2)
                    
                    # Normalize product name
                    tool_arg = self.normalize_product_name(raw_arg) if tool_name in ["query_stock_level", "get_product_price"] else raw_arg
                    
                    if tool_name not in self.tools:
                        observation = f"ERROR: Tool '{tool_name}' not available. Use: {', '.join(self.tools.keys())}"
                    else:
                        try:
                            observation = self.tools[tool_name](tool_arg)
                            print(f"✅ Tool executed: {tool_name}({tool_arg})")
                        except Exception as e:
                            observation = f"TOOL ERROR: {str(e)}"
                else:
                    observation = f"PARSING ERROR: Invalid action format '{action_text}'"
                
                print(f"🔍 Observation: {observation}")
                self.messages.append({"role": "observation", "content": observation})
                continue
            
            # No action found
            if iteration < max_iterations - 1:
                self.messages.append({"role": "user", "content": "Continue reasoning. Use Action: tool[arg] format or provide Answer."})
        
        return f"MAX ITERATIONS REACHED. Last response: {response[:100]}..."

# Initialize with notebook-friendly parameters
robust_llm = QuotaAwareLLM(
    llm=llm,
    max_retries=1,  # Reduced from 3
    sleep_on_quota_sec=5  # Reduced from 35s (notebook friendly)
)

react_agent = ReActAgent(
    robust_llm=robust_llm,
    system_prompt=REACT_SYSTEM_PROMPT,
    tools={
        "query_stock_level": query_stock_level,
        "get_product_price": get_product_price
    }
)

print("[✅] Optimized agent ready - reduced sleep times for notebook testing")

[✅] Optimized agent ready - reduced sleep times for notebook testing


In [7]:
# 9. Execution wrapper with safe defaults
def run_react_agent(query: str, max_iterations: int = 3) -> str:
    """Safe execution interface for notebook testing"""
    try:
        return react_agent.run(query, max_iterations=max_iterations)
    except Exception as e:
        return f"EXECUTION FAILED: {str(e)}"

In [10]:
# 10. Practical test suite with notebook-friendly pacing
print("="*50)
print("🧪 NOTEBOOK-OPTIMIZED REACT AGENT TESTS")
print("="*50)

test_cases = [
    ("Stock check (singular)", "How many keyboard are in stock?"),
    ("Price query", "What is the price of a headset?"),
    ("Plural handling", "Do we have monitors in stock?"),
    ("Error case", "What's the price of a chair?")
]

for title, query in test_cases:
    print(f"\n{'='*40}")
    print(f"🔹 TEST: {title}")
    print(f"❓ Query: {query}")
    
    result = run_react_agent(query, max_iterations=3)
    print(f"\n✅ RESULT: {result}")
    
    # Short pause between queries to avoid quota limits
    if test_cases.index((title, query)) < len(test_cases) - 1:
        print("\n⏳ Pausing 3s before next query...")
        time.sleep(3)

print("\n" + "="*50)
print("[🎉] TEST SUITE COMPLETED")
print("="*50)

🧪 NOTEBOOK-OPTIMIZED REACT AGENT TESTS

🔹 TEST: Stock check (singular)
❓ Query: How many keyboard are in stock?

🔍 Query: How many keyboard are in stock?

🔄 Iteration 1/3
🤖 Response: Action: query_stock_level[keyboard]
PAUSE...
⚡ Action: query_stock_level[keyboard]
✅ Tool executed: query_stock_level(keyboard)
🔍 Observation: Stock available: 120 units of Keyboard

🔄 Iteration 2/3
🤖 Response: Action: get_product_price[gaming mouse]
PAUSE...
⚡ Action: get_product_price[gaming mouse]
✅ Tool executed: get_product_price(gaming mouse)
🔍 Observation: PRICE: Gaming Mouse - $99.50 USD per unit

🔄 Iteration 3/3
🤖 Response: Answer: We have 120 units of Keyboard in stock....

✅ RESULT: We have 120 units of Keyboard in stock.

⏳ Pausing 3s before next query...

🔹 TEST: Price query
❓ Query: What is the price of a headset?

🔍 Query: What is the price of a headset?

🔄 Iteration 1/3
🤖 Response: Action: get_product_price[headset]
PAUSE...
⚡ Action: get_product_price[headset]
✅ Tool executed: get_product_

In [8]:
# 11. Agent execution with history tracking (optimized for our setup)
from typing import Tuple, List, Dict

# Global tools definition (required for agent recreation fallback)
available_tools = {
    "query_stock_level": query_stock_level,
    "get_product_price": get_product_price
}

def run_react_agent_with_history(
    question: str,
    max_iterations: int = 3  # Reduced to match our notebook-optimized agent
) -> Tuple[str, List[Dict[str, str]]]:
    """
    Execute agent with state reset per call and return answer + clean history.
    
    Key improvements:
    - Resets agent state before each run (stateless calls)
    - Uses our global quota-aware agent
    - Returns minimal history (no system prompt duplication)
    """
    global react_agent
    
    # Reset agent state for fresh execution
    react_agent.messages = [
        {"role": "system", "content": react_agent.system_prompt}
    ]
    
    # Execute with controlled iterations
    final_answer = react_agent.run(question, max_iterations=max_iterations)
    
    # Clean history: remove system prompt duplicates and internal roles
    clean_history = [
        msg for msg in react_agent.messages
        if msg["role"] not in ["system", "observation"]  # Keep only user/assistant
    ]
    
    return final_answer, clean_history

def print_full_history(history: List[Dict[str, str]]):
    """Print history with notebook-friendly formatting"""
    print("\n" + "="*60)
    print("📜 CONVERSATION HISTORY")
    print("="*60)
    
    for i, entry in enumerate(history):
        role = entry["role"].upper()
        content = entry["content"].strip()
        print(f"\n[{i+1}] {role}:")
        print(content)
        print("-"*40)
    
    print("\n" + "="*60)

print("[✅] History tracking functions ready for course exercises")

[✅] History tracking functions ready for course exercises


In [12]:
# 12. Example usage with clean output formatting
print("="*70)
print("🔍 DEMONSTRATION: Agent execution with history tracking")
print("="*70)

example_question = "How many keyboard are in stock?"
print(f"\n❓ USER QUESTION: {example_question}")

# Execute agent with history tracking
final_answer, full_history = run_react_agent_with_history(
    example_question,
    max_iterations=3  # Notebook-optimized parameter
)

# Print final answer prominently
print(f"\n{'='*50}")
print("✅ FINAL AGENT ANSWER")
print(f"{'='*50}")
print(f"➤ {final_answer}")
print(f"{'='*50}")

# Print clean interaction history
print_full_history(full_history)

print("\n💡 NOTE: History shows only user/assistant interactions (internal tool calls and system prompts are filtered for clarity)")

🔍 DEMONSTRATION: Agent execution with history tracking

❓ USER QUESTION: How many keyboard are in stock?

🔍 Query: How many keyboard are in stock?

🔄 Iteration 1/3
🤖 Response: Action: query_stock_level[keyboard]
PAUSE...
⚡ Action: query_stock_level[keyboard]
✅ Tool executed: query_stock_level(keyboard)
🔍 Observation: Stock available: 120 units of Keyboard

🔄 Iteration 2/3
🤖 Response: Action: get_product_price[gaming mouse]
PAUSE...
⚡ Action: get_product_price[gaming mouse]
✅ Tool executed: get_product_price(gaming mouse)
🔍 Observation: PRICE: Gaming Mouse - $99.50 USD per unit

🔄 Iteration 3/3
🤖 Response: Answer: We have 120 units of Keyboard in stock....

✅ FINAL AGENT ANSWER
➤ We have 120 units of Keyboard in stock.

📜 CONVERSATION HISTORY

[1] USER:
How many keyboard are in stock?
----------------------------------------

[2] ASSISTANT:
Action: query_stock_level[keyboard]
PAUSE
----------------------------------------

[3] ASSISTANT:
Action: get_product_price[gaming mouse]
PAUSE
---

In [9]:
# 13. Production-ready function to find most expensive product
def find_most_expensive_product() -> str:
    """
    Returns the name and price of the most expensive product in inventory.
    Uses existing price lookup tool to avoid data duplication.
    
    Why this implementation:
    - Reuses our validated get_product_price tool (single source of truth)
    - Handles price parsing errors gracefully
    - Maintains consistent formatting with other agent responses
    - Avoids hardcoding prices (prevents data drift)
    """
    # Products to check (matches our tool's known items)
    products_to_check = [
        "monitor", "keyboard", "gaming mouse", 
        "webcam", "headset", "printer"
    ]
    
    product_prices = {}
    
    # Get prices using our existing tool (avoids data duplication)
    for product in products_to_check:
        price_response = get_product_price(product)
        
        # Parse price from tool's response format: "PRICE: Product - $123.45 USD per unit"
        if "PRICE:" in price_response and "$" in price_response:
            try:
                # Extract price value (e.g., "$999.90" -> 999.90)
                price_str = price_response.split("$")[1].split(" ")[0]
                price_value = float(price_str.replace(",", ""))
                product_prices[product] = price_value
            except (IndexError, ValueError) as e:
                print(f"[WARNING] Failed to parse price for '{product}': {price_response}")
                continue
    
    # Safety checks
    if not product_prices:
        return "ERROR: Could not retrieve any product prices from inventory system."
    
    if len(product_prices) < len(products_to_check):
        print(f"[INFO] Retrieved prices for {len(product_prices)}/{len(products_to_check)} products")
    
    # Find most expensive product
    most_expensive = max(product_prices.items(), key=lambda x: x[1])
    product_name, price_value = most_expensive
    
    # Format with proper capitalization (handles multi-word products)
    display_name = " ".join(word.capitalize() for word in product_name.split())
    
    return (
        f"The most expensive product is '{display_name}' "
        f"with a price of ${price_value:,.2f} USD."
    )

# Quick validation test
print("[✅] Function defined. Test result:")
print(find_most_expensive_product())

[✅] Function defined. Test result:
The most expensive product is 'Monitor' with a price of $999.90 USD.


In [10]:
# 14. Course-compatible ReAct execution wrapper (fully integrated with our agent)
import re

def _normalize_product_name(name: str) -> str:
    """Normalize product names for tool execution (matches our ReActAgent logic)"""
    cleaned = name.lower().strip().replace("-", " ").replace("_", " ")
    aliases = {
        "keyboards": "keyboard",
        "mice": "mouse",
        "monitors": "monitor",
        "headsets": "headset",
        "printers": "printer",
        "laptops": "laptop",
        "webcams": "webcam"
    }
    return aliases.get(cleaned, cleaned)

def run_react_agent(question: str, max_iterations: int = 3, reset_history: bool = True) -> str:
    """
    Course-compatible wrapper that integrates with our quota-aware ReActAgent.
    
    Key adaptations:
    - Uses our existing global react_agent instance
    - Maintains quota handling and product normalization
    - Preserves course contract while avoiding logic duplication
    - Returns clean final answers compatible with course expectations
    """
    global react_agent
    
    # Reset history if requested (keep system prompt)
    if reset_history:
        react_agent.messages = [
            {"role": "system", "content": react_agent.system_prompt}
        ]
    
    # Execute using our robust agent implementation
    final_answer = react_agent.run(question, max_iterations=max_iterations)
    
    # Post-process answer to match course expectations:
    # 1. Remove prefixes like "Answer: " if present
    # 2. Handle tool errors gracefully
    if final_answer.startswith("Answer:"):
        final_answer = final_answer.split("Answer:", 1)[1].strip()
    
    # Handle common error patterns from tools
    error_patterns = [
        "not found in inventory",
        "not found in pricing catalog",
        "ERROR:",
        "PARSING ERROR:",
        "TOOL ERROR:"
    ]
    if any(pattern in final_answer.lower() for pattern in error_patterns):
        # Convert tool errors to user-friendly messages
        if "keyboard" in final_answer.lower() and "not found" in final_answer.lower():
            return "I cannot find 'keyboard' in the inventory. Please verify the product name."
        elif "not found" in final_answer.lower():
            product_match = re.search(r"'(.*?)'", final_answer)
            product = product_match.group(1) if product_match else "the product"
            return f"I cannot find '{product}' in the inventory. Please verify the product name."
    
    return final_answer

print("[✅] Course-compatible ReAct wrapper ready with quota handling and product normalization")

[✅] Course-compatible ReAct wrapper ready with quota handling and product normalization


In [15]:
# 15. CORRECTED: Tool registration + comprehensive test suite
import time

print("="*70)
print("🔧 AGENT TOOL REGISTRATION")
print("="*70)

# 1. Register new tool with the agent
print("\n[+] Registering new tool: find_most_expensive_product")
react_agent.tools["find_most_expensive_product"] = find_most_expensive_product

# 2. Update system prompt to include new tool
updated_prompt = """
You are a reasoning agent following the ReAct (Reasoning + Acting) framework.

Workflow:
1. THINK: Analyze the question and plan your approach
2. ACT: Use available tools to gather information
3. OBSERVE: Process tool responses
4. Repeat until ready for final ANSWER

Available tools:
- query_stock_level(item): Check inventory stock for a product
- get_product_price(product): Get current price of a product
- find_most_expensive_product(): Returns the most expensive product in inventory (no arguments needed)

Response format:
Thought: [Your reasoning]
Action: tool_name[argument]
PAUSE

Observation: [Tool result will appear here]
Answer: [Final comprehensive answer]

Example:
User: What's the price of a gaming mouse and how many are in stock?
Thought: I need to get the price first, then check stock levels.
Action: get_product_price[gaming mouse]
PAUSE

Observation: PRICE: Gaming Mouse - $99.50 USD per unit
Thought: Now I need to check the inventory.
Action: query_stock_level[gaming mouse]
PAUSE

Observation: Stock available: 80 units of Gaming Mouse
Answer: The gaming mouse costs $99.50 each and we have 80 units in stock.
""".strip()

react_agent.system_prompt = updated_prompt
react_agent.messages = [{"role": "system", "content": updated_prompt}]

print("[✅] Tool registration completed successfully")
print(f"Available tools: {list(react_agent.tools.keys())}")
print(f"System prompt updated with new tool description")

# 3. Run comprehensive tests
print("\n" + "="*70)
print("🧪 REACT AGENT INTEGRATION TESTS")
print("="*70)

test_cases = [
    ("Stock Check", "How many keyboard are in stock?"),
    ("Price Query", "What is the price of a headset?"),
    ("Error Handling", "Do we have chairs in stock?"),
    ("Complex Query", "What is the most expensive product?")
]

for test_name, question in test_cases:
    print(f"\n{'='*150}")
    print(f"🔹 TEST: {test_name}")
    print(f"❓ Query: {question}")
    print("-"*150)
    
    try:
        # Use notebook-optimized parameters
        answer = run_react_agent(
            question,
            max_iterations=3,
            reset_history=True
        )
        print(f"\n✅ RESULT: {answer}")
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        answer = f"EXECUTION FAILED: {str(e)}"
    
    # Strategic pause to avoid quota limits
    if test_cases.index((test_name, question)) < len(test_cases) - 1:
        print("\n⏳ Pausing 3 seconds before next query (quota management)...")
        time.sleep(3)

print(f"\n{'='*70}")
print("[🎉] ALL TESTS COMPLETED SUCCESSFULLY")
print(f"{'='*70}")

🔧 AGENT TOOL REGISTRATION

[+] Registering new tool: find_most_expensive_product
[✅] Tool registration completed successfully
Available tools: ['query_stock_level', 'get_product_price', 'find_most_expensive_product']
System prompt updated with new tool description

🧪 REACT AGENT INTEGRATION TESTS

🔹 TEST: Stock Check
❓ Query: How many keyboard are in stock?
------------------------------------------------------------------------------------------------------------------------------------------------------

🔍 Query: How many keyboard are in stock?

🔄 Iteration 1/3
🤖 Response: Action: query_stock_level[keyboard]
PAUSE...
⚡ Action: query_stock_level[keyboard]
✅ Tool executed: query_stock_level(keyboard)
🔍 Observation: Stock available: 120 units of Keyboard

🔄 Iteration 2/3
[QuotaAwareLLM] Quota error detected. Sleeping 5s (attempt 1/1)...
🤖 Response: Action: query_stock_level[keyboard]
PAUSE...
⚡ Action: query_stock_level[keyboard]
✅ Tool executed: query_stock_level(keyboard)
🔍 Observatio

In [ ]:
# 16. Interactive agent conversation loop (Jupyter-optimized)
import time
from IPython.display import clear_output

def iniciar_conversacao_com_agente():
    """
    Jupyter-optimized interactive loop with quota management.
    Features:
    - Automatic pauses between calls to avoid quota limits
    - Clear visual feedback
    - Manual history reset capability
    - Error resilience
    """
    print("="*80)
    print("💬 INTERACTIVE INVENTORY AGENT")
    print("="*80)
    print("\nAvailable commands:")
    print("- Ask any inventory question (stock levels, prices, most expensive item)")
    print("- Type 'reset' to clear conversation history")
    print("- Type 'tools' to see available functions")
    print("- Type 'exit' or 'sair' to end conversation")
    print("\n" + "*" * 80)
    
    conversation_history = []
    last_call_time = 0
    
    while True:
        # Get user input with timeout protection
        try:
            pergunta_usuario = input("\n[You]: ").strip()
        except KeyboardInterrupt:
            print("\n\n⚠️ Conversation interrupted. Type 'exit' to close properly.")
            continue
        
        # Exit condition
        if pergunta_usuario.lower() in ['exit', 'sair', 'quit', 'parar']:
            print("\n" + "="*50)
            print("👋 AGENT SESSION ENDED")
            print("="*50)
            break
        
        # Special commands
        if pergunta_usuario.lower() == 'reset':
            react_agent.messages = [{"role": "system", "content": react_agent.system_prompt}]
            conversation_history = []
            print("\n[Agent]: ✅ Conversation history reset successfully!")
            continue
        
        if pergunta_usuario.lower() == 'tools':
            tools_list = ", ".join(react_agent.tools.keys())
            print(f"\n[Agent]: 🔧 Available tools: {tools_list}")
            continue
        
        # Quota management: enforce minimum 2s between calls
        current_time = time.time()
        time_since_last_call = current_time - last_call_time
        
        if time_since_last_call < 2.0:
            wait_time = 2.0 - time_since_last_call
            print(f"\n[Agent]: ⏳ Waiting {wait_time:.1f}s to avoid quota limits...")
            time.sleep(wait_time)
        
        print("\n[Agent]: 🔄 Processing your request...")
        
        try:
            # Execute agent with optimized parameters
            resposta_agente = run_react_agent(
                pergunta_usuario,
                max_iterations=4,  # Slightly higher for complex questions
                reset_history=False  # Maintain conversation context
            )
            
            # Record interaction
            conversation_history.append({
                "user": pergunta_usuario,
                "agent": resposta_agente
            })
            
            # Display response with clear formatting
            print("\n" + "-"*80)
            print(f"[Agent]: {resposta_agente}")
            print("-"*80)
            
            last_call_time = time.time()
            
        except Exception as e:
            error_msg = str(e)
            if "quota" in error_msg.lower() or "rate limit" in error_msg.lower():
                print(f"\n[Agent]: ⚠️ QUOTA LIMIT REACHED. Please wait 5 seconds before next question.")
                time.sleep(5)
            else:
                print(f"\n[Agent]: ❌ ERROR: {error_msg}")
                print("Try rephrasing your question or type 'reset' to clear history.")
        
        print("\n💡 Tip: Type 'exit' to end conversation, 'reset' to clear history, or 'tools' to see available functions")

# Start interactive session
if __name__ == "__main__":
    iniciar_conversacao_com_agente()

print("\n[✅] Interactive agent module loaded successfully")
print("To start conversation, run: iniciar_conversacao_com_agente()")

💬 INTERACTIVE INVENTORY AGENT

Available commands:
- Ask any inventory question (stock levels, prices, most expensive item)
- Type 'reset' to clear conversation history
- Type 'tools' to see available functions
- Type 'exit' or 'sair' to end conversation

********************************************************************************



[You]:  Give me a list of products in stock



[Agent]: 🔄 Processing your request...

🔍 Query: Give me a list of products in stock

🔄 Iteration 1/4
🤖 Response: Thought: The user is asking for a list of products in stock. I have tools to `query_stock_level` for a specific item and `get_product_price` for a specific product. However, I do not have a tool that ...

--------------------------------------------------------------------------------
[Agent]: I can check the stock level for a specific product if you tell me its name, but I don't have a tool to list all products currently in stock.
--------------------------------------------------------------------------------

💡 Tip: Type 'exit' to end conversation, 'reset' to clear history, or 'tools' to see available functions

[Agent]: ✅ Conversation history reset successfully!



[You]:  tools



[Agent]: 🔧 Available tools: query_stock_level, get_product_price



[You]:  What's the cheapest product?



[Agent]: 🔄 Processing your request...

🔍 Query: What's the cheapest product?

🔄 Iteration 1/4
🤖 Response: Thought: The user is asking for the "cheapest product". To determine the cheapest product, I would need a list of all available products and their respective prices. However, I do not have a tool that...

--------------------------------------------------------------------------------
[Agent]: I cannot determine the cheapest product because I do not have a tool to list all available products and their prices. I can only check the price or stock level of a specific product if you tell me its name.
--------------------------------------------------------------------------------

💡 Tip: Type 'exit' to end conversation, 'reset' to clear history, or 'tools' to see available functions



[You]:  i dont remember the item in the list, could you show me the avaliable products?



[Agent]: 🔄 Processing your request...

🔍 Query: i dont remember the item in the list, could you show me the avaliable products?

🔄 Iteration 1/4
[QuotaAwareLLM] Quota error detected. Sleeping 5s (attempt 1/1)...
[!] LLM error: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 52.792440193s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goog

In [ ]:
iniciar_conversacao_com_agente()

💬 INTERACTIVE INVENTORY AGENT

Available commands:
- Ask any inventory question (stock levels, prices, most expensive item)
- Type 'reset' to clear conversation history
- Type 'tools' to see available functions
- Type 'exit' or 'sair' to end conversation

********************************************************************************



[You]:  make a list with our stock products



[Agent]: 🔄 Processing your request...

🔍 Query: make a list with our stock products

🔄 Iteration 1/4
[QuotaAwareLLM] Quota error detected. Sleeping 5s (attempt 1/1)...
[!] LLM error: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 57.723367251s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violat


[You]:  reset
